In [1]:
import random
import sqlite3

def extract_from_db(cursor, id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

def extract_random_from_db(cursor, include_parent=True):
    cursor.execute("SELECT * FROM laws WHERE title LIKE '%Điểm%'")
    rows = cursor.fetchall()
    if not rows:
        return []

    columns = [col[0] for col in cursor.description]
    row = random.choice(rows)
    result = dict(zip(columns, row))

    results = [result]

    if include_parent:
        current = result
        while current['parent_id'] is not None:
            cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
            parent = cursor.fetchone()
            if parent is None:
                break
            parent_dict = dict(zip(columns, parent))
            results.append(parent_dict)
            current = parent_dict

    return results[::-1]

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

def get_gpt_response(user_prompt, system_prompt, api_key, model="gpt-4o"):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [2]:
def clean_text(text: str) -> str:
    """Remove newlines, tabs, extra spaces, punctuation and lowercase."""
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[!?]+", "", text)
    return text.strip().lower()

In [3]:
conn = sqlite3.connect(r"E:\Github\LawAssistant\triplet_extraction\law.db")
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Các bảng trong law.db:", tables)

Các bảng trong law.db: [('laws',), ('law_refs',)]


In [7]:
rows = extract_random_from_db(cursor, True)
law = ""
title = ""
for r in rows:
    title += r['title'] + " "
    if not (r['title'].strip().startswith("Chương") or  r['title'].strip().startswith("Điều")):
        law += r['content'] + "\n"

so_hieu = r['so_hieu']
id = r['id']
print(id)
print(so_hieu + " " + title)
print(clean_text(law))

b4988f41df11b72ebef60ea88658942dfe5de8d3482c03a83c0f46d3f4349c62
36/2024/QH15 Chương III Điều 52 Khoản 3 Điểm b 
xe quá khổ giới hạn, xe quá tải trọng, xe bánh xích được cấp giấy phép lưu hành xe trên đường bộ trong các trường hợp sau đây: lưu hành xe quá khổ giới hạn, xe quá tải trọng để chở hàng hóa trên đường bộ trong các trường hợp: phục vụ nhiệm vụ quốc phòng, an ninh; phòng, chống, khắc phục hậu quả thiên tai; thực hiện nhiệm vụ trong trường hợp khẩn cấp; chở hàng siêu trường, siêu trọng khi các phương thức vận chuyển hàng hoá bằng đường sắt, đường thuỷ nội địa, hàng không, hàng hải không phù hợp hoặc phải kết hợp phương thức vận tải đường bộ với phương thức vận tải khác;


In [6]:
system_prompt_rewrite = """
Bạn là trợ lý AI Tiếng Việt chuyên nghiệp và trung thực.
Bạn là chuyên gia pháp luật Việt Nam, am hiểu các bộ luật, nghị định, và văn bản pháp luật.
Bạn là chuyên gia ngôn ngữ Việt Nam, biết viết câu chuẩn cấu trúc, chính xác, trang trọng, và đúng ngôn ngữ pháp lý.
Luôn trả lời chính xác, hữu ích, ngắn gọn và an toàn.
Nếu thông tin không hợp lý hoặc thiếu, hãy yêu cầu thêm thông tin thay vì đoán mò.
Không thay đổi ý nghĩa khi viết lại câu.
Luôn dùng ngôn ngữ chính xác như trong văn bản pháp luật, tránh ngôn ngữ thông thường hay không trang trọng.

Định nghĩa 'câu đơn': Một câu đơn là câu có một chủ ngữ (hoặc cụm chủ ngữ) và một vị ngữ (hoặc cụm vị ngữ), biểu đạt một ý trọn vẹn; câu có thể chứa thành tố phụ (tính từ, trạng từ, bổ ngữ) nhưng không được ghép bằng liên từ hoặc dấu câu như dấu ",", ";" để tạo hai hoặc nhiều mệnh đề độc lập.

Quy tắc bắt buộc:
1. Khi viết lại, **chỉ** trả về các câu đơn theo đúng định nghĩa trên; mỗi câu một dòng nếu có nhiều câu.
2. **Được phép tái sử dụng** các thành phần câu (chủ ngữ, cụm danh từ, đại từ, cụm tính từ, v.v.) từ vế trước hoặc từ phần khác của câu gốc để hoàn chỉnh vế thiếu, **nhằm bảo toàn ý nghĩa** sau khi tách.
3. Khi tái sử dụng, **ưu tiên giữ nguyên** từ ngữ gốc; chỉ thực hiện điều chỉnh nhỏ cần thiết để tạo câu đơn ngữ pháp đúng, **không** thêm thông tin, suy đoán hay nội dung mới.
4. Tuyệt đối không kèm chú giải, giải thích, danh sách hay bất kỳ nội dung nào khác ngoài các câu viết lại.
5. Nếu câu gốc mơ hồ hoặc thiếu thông tin đến mức không thể tạo câu đơn hoàn chỉnh mà vẫn giữ nguyên ý, hãy yêu cầu thêm thông tin ngắn gọn.
Danh mục liên từ cần loại trừ khi viết câu đơn: và, hoặc, hoặc là, hay, hay là, nhưng, song, tuy nhiên, mà, còn, rồi.
Giữ nguyên thứ tự trước sau của các từ sau khi viết lại câu.
Sau khi viết lại câu không được thiếu từ danh từ nào trong câu gốc và phải độc lập không phụ thuộc vào câu trước đó.
"""


user_prompt_rewrite = f"""
Ngữ cảnh: Bộ luật số {so_hieu} trong luật Việt Nam
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ và vị ngữ, giữ nguyên ý nghĩa. Mỗi câu xuất ra phải là một câu đơn đầy đủ (một dòng một câu nếu có nhiều câu).
Câu cần viết lại: "{clean_text(law)}"
"""


In [32]:
print(user_prompt_rewrite)


Ngữ cảnh: Bộ luật số 36/2024/QH15 trong luật Việt Nam
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ và vị ngữ, giữ nguyên ý nghĩa. Mỗi câu xuất ra phải là một câu đơn đầy đủ (một dòng một câu nếu có nhiều câu).
Câu cần viết lại: "người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát, giảm tốc độ hoặc dừng lại để bảo đảm an toàn trong các trường hợp sau đây: nơi đường bộ giao nhau cùng mức với đường bộ, đường bộ giao nhau cùng mức với đường sắt; đường hẹp, đường vòng, đường quanh co, đường đèo, dốc;"



In [33]:
rewrite_sentence = get_gpt_response(user_prompt_rewrite, system_prompt_rewrite, api_key, "gpt-4.1-mini")

Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại nơi đường bộ giao nhau cùng mức với đường bộ.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại nơi đường bộ giao nhau cùng mức với đường bộ.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại nơi đường bộ giao nhau cùng mức với đường bộ.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại nơi đường bộ giao nhau cùng mức với đường sắt.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại nơi đường bộ giao nhau cùng mức với đường sắt.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải dừng lại để bảo đảm an toàn tại nơi đường bộ giao nhau cùng mức với đường sắt.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải quan sát tại đường hẹp.  
Người điều khiển phương tiện tham gia giao thông đường bộ phải giảm tốc độ tại đường hẹp.  
Người điều khiển phương tiện

In [7]:
import re

vncorenlp_pos_map = {
    "N":   "Noun (Danh từ)",
    "Np":  "Proper noun (Danh từ riêng)",
    "Nc":  "Classifier noun (Danh từ giống loại)",
    "Nu":  "Unit noun (Danh từ đơn vị)",
    "V":   "Verb (Động từ)",
    "Vb":  "Verb (base) (Động từ gốc)",
    "A":   "Adjective (Tính từ)",
    "Ai":  "Adjective (predicative) (Tính từ vị ngữ)",
    "P":   "Pronoun (Đại từ)",
    "R":   "Adverb (Trạng từ)",
    "M":   "Numeral / number (Số từ)",
    "E":   "Preposition / particle (Giới từ / trợ từ)",
    "C":   "Coordinating conjunction (Liên từ phối hợp)",
    "CC":  "Subordinating conjunction / complementizer (Liên từ phụ thuộc)",
    "L":   "Determiner / article (Từ hạn định)",
    "D":   "Adverbial marker / degree marker (Từ chỉ mức độ)",
    "X":   "Other (Khác)",
    "CH":  "Punctuation (Dấu câu)"
}

def process_sentence(text: str, rdrsegmenter, verbose: bool = True) -> dict:
    results = {
        "original": text,
        "cleaned": "",
        "segmented": [],
        "pos_annotation": [],
        "concepts": []
    }

    if verbose:
        print("1. Original text:\n", text, "\n")

    # 2. Clean
    text = clean_text(text)
    results["cleaned"] = text
    if verbose:
        print("2. Cleaned text:\n", text, "\n")

    # 3. Segmentation
    segmented = rdrsegmenter.word_segment(text)
    results["segmented"] = segmented
    if verbose:
        print("3. Segmented text (tokens):\n", segmented, "\n")

    # 4. POS tagging
    output = rdrsegmenter.annotate_text(text)
    sents = output.values() if isinstance(output, dict) else output

    pos_annot = []
    tokens = []

    if verbose:
        print("4. POS annotation:")
        print(f"{'Idx':<5} {'Token':<15} {'POS':<20}")
        print("-" * 45)

    for sent in sents:
        if not isinstance(sent, list):
            continue
        for token in sent:
            if not isinstance(token, dict):
                continue
            word = token.get("wordForm", "")
            pos = token.get("posTag", "")
            pos_full = vncorenlp_pos_map.get(pos, pos)

            pos_data = {
                "index": token.get("index", ""),
                "token": word,
                "pos": pos_full
            }
            pos_annot.append(pos_data)

            if verbose:
                print(f"{pos_data['index']:<5} {pos_data['token']:<15} {pos_data['pos']:<20}")

            # Only append N, V, A
            if pos.startswith(("N", "V", "A")):
                tokens.append((word.replace("_", " "), pos))

    results["pos_annotation"] = pos_annot

    # 5. Triplet extraction
    concepts = []
    concept1_tokens = []
    concept2_tokens = []
    relation_tokens = []

    for token, pos in tokens:
        if pos.startswith("V"):
            if concept2_tokens:
                triplet = (
                    " ".join(concept1_tokens),
                    " ".join(relation_tokens),
                    " ".join(concept2_tokens)
                )
                if triplet not in concepts:  # chỉ thêm nếu chưa tồn tại
                    concepts.append(triplet)

                concept1_tokens = concept2_tokens
                concept2_tokens = []
                relation_tokens = []

            relation_tokens.append(token)
        else:
            if relation_tokens:
                concept2_tokens.append(token)
            else:
                concept1_tokens.append(token)

    # Append last triplet if complete
    if concept1_tokens and relation_tokens and concept2_tokens:
        concepts.append((
            " ".join(concept1_tokens),
            " ".join(relation_tokens),
            " ".join(concept2_tokens)
        ))

    results["concepts"] = concepts

    if verbose:
        print("\n5. Extracted triplets:")
        for c in concepts:
            print(c)

    return results

In [8]:
def insert_triplet(tx, concept1, relation, concept2, c1_metadata=None, c2_metadata=None, rel_metadata=None):
    query = """
    MERGE (c1:Concept {name: $concept1})
    ON CREATE SET c1 = $c1_metadata
    ON MATCH SET
        c1.document_number = apoc.coll.toSet(coalesce(c1.document_number, []) + [$c1_metadata.document_number]),
        c1.title           = apoc.coll.toSet(coalesce(c1.title, []) + [$c1_metadata.title]),
        c1.document_id     = apoc.coll.toSet(coalesce(c1.document_id, []) + [$c1_metadata.document_id])

    MERGE (c2:Concept {name: $concept2})
    ON CREATE SET c2 = $c2_metadata
    ON MATCH SET
        c2.document_number = apoc.coll.toSet(coalesce(c2.document_number, []) + [$c2_metadata.document_number]),
        c2.title           = apoc.coll.toSet(coalesce(c2.title, []) + [$c2_metadata.title]),
        c2.document_id     = apoc.coll.toSet(coalesce(c2.document_id, []) + [$c2_metadata.document_id])

    MERGE (c1)-[r:RELATION {name: $relation}]->(c2)
    ON CREATE SET r = $rel_metadata
    ON MATCH SET  r += $rel_metadata

    RETURN c1, r, c2
    """
    tx.run(
        query,
        concept1=concept1,
        concept2=concept2,
        relation=relation,
        c1_metadata=c1_metadata or {},
        c2_metadata=c2_metadata or {},
        rel_metadata=rel_metadata or {}
    )

def delete_all(tx):
    # Delete all nodes and relationships
    tx.run("MATCH (n) DETACH DELETE n")

In [9]:
def extract_triplet(input_text: str, rdrsegmenter, driver, db_name, document_id, document_number):
    with driver.session(database=db_name) as session:
        split_text = input_text.split("\n")
        for t in split_text:
            res = process_sentence(t, rdrsegmenter, verbose=False)
            for concept1, relation, concept2 in res["concepts"]:
                session.execute_write(
                    insert_triplet,
                    concept1,
                    relation,
                    concept2,
                    c1_metadata={"document_number": document_number, "document_id": document_id},
                    c2_metadata={"document_number": document_number, "document_id": document_id},
                    rel_metadata={"document_number": document_number, "document_id": document_id},
                )

In [11]:
if "rdrsegmenter" not in globals():
    import py_vncorenlp
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg", "pos"],
        save_dir=r"E:\Github\LawAssistant\triplet_extraction\VnCoreNLP-master"
    )

In [12]:
from neo4j import GraphDatabase

uri = "neo4j://127.0.0.1:7687"
username = "neo4j"
password = "1234567890"

driver = GraphDatabase.driver(uri, auth=(username, password))

In [ ]:
extract_triplet(rewrite_sentence, rdrsegmenter, driver, "ontology", id, so_hieu)

In [139]:
with driver.session(database="ontology") as session:
    session.execute_write(delete_all)